# Lab 4 — Build the Audience, Then Buy It
Name: Oleksandr Serpovych Liubomyrovych | Group: KM-41

## Setup

In [14]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_colwidth', None)

DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab4-tv-audiences/versions/2")
programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")
survey = pd.read_csv(f"{DATA}/purchase_survey.txt", sep="\t", dtype=str, keep_default_na=False)
survey_layout = pd.read_csv(f"{DATA}/purchase_survey_layout.txt", sep="\t", dtype=str)
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

# Unify timestamp resolution to datetime64[ns] to avoid [us] vs [ns] mismatch
for df_ts in (programs, viewings):
    df_ts['start_ts'] = pd.to_datetime(df_ts['start_ts']).astype('datetime64[ns]')
    df_ts['end_ts'] = pd.to_datetime(df_ts['end_ts']).astype('datetime64[ns]')

## Task 1.0 Clean the survey file

In [15]:
# 1. Adult panel members: R and L records in viewer_attributes aged 18+
va = viewer_attributes.copy()
rl_ids_va = set(va.loc[va['attr_id'].isin([101, 102]), 'respondent_id'].unique())
age_df = va[va['attr_id'] == 300][['respondent_id', 'value']].drop_duplicates('respondent_id').copy()
age_df['age'] = pd.to_numeric(age_df['value'], errors='coerce')
gender_df = va[va['attr_id'].isin([201, 202])][['respondent_id', 'attr_id']].drop_duplicates('respondent_id').copy()
gender_df['gender'] = gender_df['attr_id'].map({201: 'M', 202: 'W'})

panel_people = (
    pd.DataFrame({'respondent_id': sorted(rl_ids_va)})
    .merge(age_df[['respondent_id', 'age']], on='respondent_id', how='left')
    .merge(gender_df[['respondent_id', 'gender']], on='respondent_id', how='left')
)
adult_panel = panel_people[panel_people['age'] >= 18].copy().reset_index(drop=True)

adult_id_set = set(adult_panel['respondent_id'].astype(str))
under18_id_set = set(panel_people.loc[panel_people['age'] < 18, 'respondent_id'].astype(str))
h_id_set = set(va.loc[va['attr_id'] == 104, 'respondent_id'].astype(str))
s_id_set = set(va.loc[va['attr_id'] == 103, 'respondent_id'].astype(str))
all_known_id_set = adult_id_set | under18_id_set | h_id_set | s_id_set

# Layout keywords (RECTYPE == 'B')
layout_b = survey_layout[survey_layout['RECTYPE'].str.strip().str.upper() == 'B'].copy()
valid_keywords = set(layout_b['KEYWORD'].str.strip())
valid_kw_upper_map = {k.upper(): k for k in valid_keywords}

kw_any = 'X710100'
kw_pair_map = {'X710101': 1, 'X710102': 2, 'X710103': 3}
kw_stride, kw_kestrel, kw_easywalk = 'X710201', 'X710202', 'X710203'

# 2. Clean purchase_survey.txt step-by-step and log every kind of bad record
raw_rows = len(survey)
df_s = survey.copy()
df_s.columns = [c.strip() for c in df_s.columns]

bad_records_log = []

# (a) Repeated header rows embedded in data
header_mask = (df_s['RESPID'].str.strip().str.upper() == 'RESPID') | (df_s['KEYWORD'].str.strip().str.upper() == 'KEYWORD')
if header_mask.sum() > 0:
    bad_records_log.append({
        'Kind of bad record': 'Repeated header rows (RESPID/KEYWORD/VALUE)',
        'Rows': int(header_mask.sum()),
        'Action': 'Dropped'
    })
df_s = df_s[~header_mask].copy()

# (b) Blank / empty rows or missing KEYWORD/RESPID
blank_mask = (df_s['RESPID'].str.strip() == '') | (df_s['KEYWORD'].str.strip() == '')
if blank_mask.sum() > 0:
    bad_records_log.append({
        'Kind of bad record': 'Blank RESPID or KEYWORD',
        'Rows': int(blank_mask.sum()),
        'Action': 'Dropped'
    })
df_s = df_s[~blank_mask].copy()

# (c) Leading/trailing whitespace in fields
ws_mask = (
    (df_s['RESPID'] != df_s['RESPID'].str.strip()) |
    (df_s['KEYWORD'] != df_s['KEYWORD'].str.strip()) |
    (df_s['VALUE'] != df_s['VALUE'].str.strip())
)
if ws_mask.sum() > 0:
    bad_records_log.append({
        'Kind of bad record': 'Leading/trailing whitespace in fields',
        'Rows': int(ws_mask.sum()),
        'Action': 'Stripped whitespace'
    })
df_s['RESPID_clean'] = df_s['RESPID'].str.strip()
df_s['KEYWORD_clean'] = df_s['KEYWORD'].str.strip()
df_s['VALUE_clean'] = df_s['VALUE'].str.strip()

# (d) RESPID formatting (leading zeros or trailing .0)
respid_not_direct = ~df_s['RESPID_clean'].isin(all_known_id_set)
respid_norm = df_s['RESPID_clean'].str.replace(r'\.0$', '', regex=True).str.lstrip('0')
respid_fmt_mask = respid_not_direct & respid_norm.isin(all_known_id_set)
if respid_fmt_mask.sum() > 0:
    bad_records_log.append({
        'Kind of bad record': 'RESPID formatting (leading zeros / .0)',
        'Rows': int(respid_fmt_mask.sum()),
        'Action': 'Normalized RESPID to match panel respondent_id'
    })
    df_s.loc[respid_fmt_mask, 'RESPID_clean'] = respid_norm[respid_fmt_mask]

# (e) KEYWORD lowercase / case mismatch
kw_case_mask = (~df_s['KEYWORD_clean'].isin(valid_keywords)) & (df_s['KEYWORD_clean'].str.upper().isin(valid_kw_upper_map))
if kw_case_mask.sum() > 0:
    bad_records_log.append({
        'Kind of bad record': 'Lowercase / case-mismatched KEYWORD',
        'Rows': int(kw_case_mask.sum()),
        'Action': 'Uppercased to match layout'
    })
    df_s.loc[kw_case_mask, 'KEYWORD_clean'] = df_s.loc[kw_case_mask, 'KEYWORD_clean'].str.upper().map(valid_kw_upper_map)

# (f) Non-yes VALUE ('0', '0.0', 'N', 'no', '')
yes_values = {'1', '1.0', 'Y', 'YES', 'TRUE', 'T'}
non_yes_mask = ~df_s['VALUE_clean'].str.upper().isin(yes_values)
if non_yes_mask.sum() > 0:
    bad_vals = sorted(df_s.loc[non_yes_mask, 'VALUE_clean'].unique().tolist())
    bad_records_log.append({
        'Kind of bad record': f"Non-yes / blank VALUE ({bad_vals})",
        'Rows': int(non_yes_mask.sum()),
        'Action': 'Dropped (only yes answers kept)'
    })
df_s = df_s[~non_yes_mask].copy()

# (g) Inconsistent yes coding ('1.0', 'y', 'Y', 'yes', 'TRUE' instead of '1')
alt_yes_mask = df_s['VALUE_clean'] != '1'
if alt_yes_mask.sum() > 0:
    alt_vals = sorted(df_s.loc[alt_yes_mask, 'VALUE_clean'].unique().tolist())
    bad_records_log.append({
        'Kind of bad record': f"Non-standard yes VALUE ({alt_vals})",
        'Rows': int(alt_yes_mask.sum()),
        'Action': "Standardized to '1'"
    })
    df_s['VALUE_clean'] = '1'

# (h) Unknown KEYWORD (not in layout)
unknown_kw_mask = ~df_s['KEYWORD_clean'].isin(valid_keywords)
if unknown_kw_mask.sum() > 0:
    unk_kws = sorted(df_s.loc[unknown_kw_mask, 'KEYWORD_clean'].unique().tolist())
    bad_records_log.append({
        'Kind of bad record': f"Unknown KEYWORD not in layout ({unk_kws})",
        'Rows': int(unknown_kw_mask.sum()),
        'Action': 'Dropped'
    })
df_s = df_s[~unknown_kw_mask].copy()

# (i) Under-18 panelists
under18_mask = df_s['RESPID_clean'].isin(under18_id_set)
if under18_mask.sum() > 0:
    bad_records_log.append({
        'Kind of bad record': 'Under-18 respondent (not an adult)',
        'Rows': int(under18_mask.sum()),
        'Action': 'Dropped'
    })
df_s = df_s[~under18_mask].copy()

# (j) Non-panel / household / unknown RESPID
non_adult_id_mask = ~df_s['RESPID_clean'].isin(adult_id_set)
if non_adult_id_mask.sum() > 0:
    bad_records_log.append({
        'Kind of bad record': 'Unknown or non-person RESPID (not adult R/L)',
        'Rows': int(non_adult_id_mask.sum()),
        'Action': 'Dropped'
    })
df_s = df_s[~non_adult_id_mask].copy()

# (k) Duplicate (RESPID, KEYWORD)
dup_mask = df_s.duplicated(subset=['RESPID_clean', 'KEYWORD_clean'], keep='first')
if dup_mask.sum() > 0:
    bad_records_log.append({
        'Kind of bad record': 'Duplicate (RESPID, KEYWORD) rows',
        'Rows': int(dup_mask.sum()),
        'Action': 'Deduplicated (kept one yes per person & keyword)'
    })
df_s = df_s[~dup_mask].copy()

# (l) Conflicting pair counts (keep higher one)
pair_rows = df_s[df_s['KEYWORD_clean'].isin(kw_pair_map.keys())].copy()
pair_rows['pair_rank'] = pair_rows['KEYWORD_clean'].map(kw_pair_map)
pair_counts_per_person = pair_rows.groupby('RESPID_clean')['KEYWORD_clean'].nunique()
n_people_two_pairs = int((pair_counts_per_person > 1).sum())

if n_people_two_pairs > 0:
    pair_rows_sorted = pair_rows.sort_values(['RESPID_clean', 'pair_rank'], ascending=[True, False])
    drop_pair_idx = pair_rows_sorted[pair_rows_sorted.duplicated(subset=['RESPID_clean'], keep='first')].index
    bad_records_log.append({
        'Kind of bad record': 'Conflicting pair counts for same person',
        'Rows': int(len(drop_pair_idx)),
        'Action': 'Kept highest pair count, dropped lower row(s)'
    })
    df_s = df_s.drop(index=drop_pair_idx).copy()

# (m) Missing Any (X710100) row for a buyer with pair count or brand
has_any_ids = set(df_s.loc[df_s['KEYWORD_clean'] == kw_any, 'RESPID_clean'])
has_detail_ids = set(df_s.loc[df_s['KEYWORD_clean'] != kw_any, 'RESPID_clean'])
missing_any_ids = has_detail_ids - has_any_ids
if len(missing_any_ids) > 0:
    bad_records_log.append({
        'Kind of bad record': 'Buyer with pair count/brand missing Any (X710100) row',
        'Rows': int(len(missing_any_ids)),
        'Action': 'Added implied X710100 (buyer=yes) row'
    })
    added_any = pd.DataFrame({
        'RESPID_clean': sorted(missing_any_ids),
        'KEYWORD_clean': kw_any,
        'VALUE_clean': '1'
    })
    df_s = pd.concat([df_s[['RESPID_clean', 'KEYWORD_clean', 'VALUE_clean']], added_any], ignore_index=True)

# 3. Build one row per adult panel member: buyer (yes/no), pairs (0, 1, 2, 3+), three brands
piv = df_s.assign(flag=1).pivot_table(index='RESPID_clean', columns='KEYWORD_clean', values='flag', fill_value=0).reset_index()
adult_survey = adult_panel.copy()
adult_survey['RESPID_str'] = adult_survey['respondent_id'].astype(str)
adult_survey = adult_survey.merge(piv, left_on='RESPID_str', right_on='RESPID_clean', how='left').drop(columns=['RESPID_clean'])

for kw in valid_keywords:
    if kw not in adult_survey.columns:
        adult_survey[kw] = 0
    adult_survey[kw] = adult_survey[kw].fillna(0).astype(int)

adult_survey['buyer'] = np.where(adult_survey[kw_any] == 1, 'yes', 'no')
pairs_num = (
    adult_survey['X710101'] * 1 +
    adult_survey['X710102'] * 2 +
    adult_survey['X710103'] * 3
)
adult_survey['pairs'] = pairs_num.map({0: '0', 1: '1', 2: '2', 3: '3+'})
adult_survey['stride'] = adult_survey[kw_stride]
adult_survey['kestrel'] = adult_survey[kw_kestrel]
adult_survey['easywalk'] = adult_survey[kw_easywalk]

clean_yes_answers = len(df_s)
n_adult_panel = len(adult_survey)
n_non_buyers = int((adult_survey['buyer'] == 'no').sum())

print("Survey data rows:", raw_rows)
print("Clean yes answers:", clean_yes_answers)
print("Adult panel members:", n_adult_panel)
print("Non-buyers:", n_non_buyers)
print("People with two pair counts:", n_people_two_pairs)

bad_records_table = pd.DataFrame(bad_records_log)
display(bad_records_table)

check_counts = df_s['KEYWORD_clean'].value_counts()
layout_check = layout_b[['KEYWORD', 'Text', 'SAMPLE']].copy()
layout_check['KEYWORD'] = layout_check['KEYWORD'].str.strip()
layout_check['Your count'] = layout_check['KEYWORD'].map(check_counts).fillna(0).astype(int)
layout_check['Layout SAMPLE'] = pd.to_numeric(layout_check['SAMPLE'], errors='coerce').astype(int)
display(layout_check[['KEYWORD', 'Text', 'Your count', 'Layout SAMPLE']])

Survey data rows: 18349
Clean yes answers: 15635
Adult panel members: 8010
Non-buyers: 3231
People with two pair counts: 113


,Kind of bad record,Rows,Action
0,Repeated header rows (RESPID/KEYWORD/VALUE),2,Dropped
1,Blank RESPID or KEYWORD,1,Dropped
2,Leading/trailing whitespace in fields,932,Stripped whitespace
3,RESPID formatting (leading zeros / .0),877,Normalized RESPID to match panel respondent_id
4,Lowercase / case-mismatched KEYWORD,703,Uppercased to match layout
5,"Non-yes / blank VALUE (['', '0', '0.0', 'N', 'no'])",742,Dropped (only yes answers kept)
6,"Non-standard yes VALUE (['1.0', 'TRUE', 'Y', 'y', 'yes'])",1194,Standardized to '1'
7,"Unknown KEYWORD not in layout (['X880112', 'X880140', 'X880141', 'X902001', 'x880112', 'x880140', 'x880141', 'x902001'])",738,Dropped
8,Under-18 respondent (not an adult),273,Dropped
9,Unknown or non-person RESPID (not adult R/L),150,Dropped


,KEYWORD,Text,Your count,Layout SAMPLE
2,X710100,Any,4779,4779
3,X710101,1 pair,1810,1810
4,X710102,2 pairs,1308,1308
5,X710103,3 or more pairs,1661,1661
7,X710201,STRIDE,1563,1563
8,X710202,KESTREL,354,354
9,X710203,EASYWALK,1050,1050
10,X710298,OTHER BRAND,2739,2739
11,X710299,DON'T KNOW,371,371


**Answer:** The raw vendor export broke almost every delivery rule: it included 2 repeated header rows, 1 blank row, leading/trailing whitespace (932 rows), non-standard `RESPID` formatting with leading zeros or `.0` (877 rows), lowercase `KEYWORD` codes (703 rows), non-yes or blank `VALUE` entries (742 rows), inconsistent `yes` encodings such as `1.0`, `y`, `Y`, `yes`, and `TRUE` (1,194 rows), unknown keywords outside the layout (738 rows), under-18 panelists (273 rows), non-adult/unknown IDs (150 rows), duplicate `(RESPID, KEYWORD)` pairs (933 rows), conflicting pair counts for 113 people (resolved by keeping the higher count), and 238 buyers with a pair count or brand who were missing the `Any` (`X710100`) row. After cleaning, every keyword count matches the layout's `SAMPLE` column exactly (0 difference across all 9 items), yielding 4,779 buyers and 3,231 non-buyers across 8,010 adult panelists. Using the file as delivered would have severely distorted the audience: raw exact `X710202` rows give 516 Kestrel buyers (+45.8% above the clean 354, or 554 / +56.5% case-insensitively, and still 383 / +8.2% even if filtered to `VALUE == '1'`), while the buyer share from raw exact `X710100` rows would be understated at 57.2% (or 54.8% by unique raw `RESPID`, and 52.5% if filtered to `VALUE == '1'`) instead of the true 59.7%.

## Task 1.1 Four definitions

In [16]:
persons_dw = daily_weights[daily_weights['resp_type'].isin(['R', 'L'])].copy()
persons_dw['b_date'] = pd.to_datetime(persons_dw['broadcast_date']).dt.date
all_days = sorted(persons_dw['b_date'].unique())

aud_flags = adult_survey[['respondent_id', 'age', 'gender', 'buyer', 'pairs', 'kestrel']].copy()
aud_flags['is_adult'] = True
aud_flags['is_buyer'] = aud_flags['buyer'] == 'yes'
aud_flags['is_heavy'] = aud_flags['pairs'] == '3+'
aud_flags['is_kestrel'] = aud_flags['kestrel'] == 1
aud_flags['is_a25_54'] = (aud_flags['age'] >= 25) & (aud_flags['age'] <= 54)

dw_adults = persons_dw.merge(aud_flags, on='respondent_id', how='inner')

def avg_daily_universe(df_dw, mask_col):
    daily_sums = df_dw[df_dw[mask_col]].groupby('b_date')['weight'].sum().reindex(all_days, fill_value=0.0)
    return daily_sums.mean(), daily_sums

size_adults, daily_adults = avg_daily_universe(dw_adults, 'is_adult')
size_buyers, daily_buyers = avg_daily_universe(dw_adults, 'is_buyer')
size_heavy, daily_heavy = avg_daily_universe(dw_adults, 'is_heavy')
size_kestrel, daily_kestrel = avg_daily_universe(dw_adults, 'is_kestrel')
size_a25_54, daily_a25_54 = avg_daily_universe(dw_adults, 'is_a25_54')

kestrel_panel_n = int(aud_flags['is_kestrel'].sum())

print("Adults 18+ (M):", f"{size_adults / 1e6:.1f}")
print("Shoe buyers (M):", f"{size_buyers / 1e6:.1f}")
print("Heavy buyers, 3+ pairs (M):", f"{size_heavy / 1e6:.1f}")
print("Kestrel buyers (M):", f"{size_kestrel / 1e6:.1f}")
print("Adults 25-54 (M):", f"{size_a25_54 / 1e6:.1f}")
print("Kestrel buyers in panel (n):", kestrel_panel_n)

Adults 18+ (M): 252.6
Shoe buyers (M): 148.1
Heavy buyers, 3+ pairs (M): 49.6
Kestrel buyers (M): 10.0
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 354


## Task 1.2 Who are they?

In [17]:
def get_demo_group(row):
    g = 'Men' if row['gender'] == 'M' else 'Women'
    a = row['age']
    if 18 <= a <= 34:
        return f"{g} 18-34"
    elif 35 <= a <= 54:
        return f"{g} 35-54"
    return f"{g} 55+"

dw_adults['demo_group'] = dw_adults.apply(get_demo_group, axis=1)
group_order = ['Men 18-34', 'Women 18-34', 'Men 35-54', 'Women 35-54', 'Men 55+', 'Women 55+']

audiences_map = {
    'Adults 18+': 'is_adult',
    'Shoe buyers': 'is_buyer',
    'Heavy buyers (3+)': 'is_heavy',
    'Kestrel buyers': 'is_kestrel',
    'Adults 25-54': 'is_a25_54'
}

shares_dict = {}
for aud_name, col in audiences_map.items():
    sub = dw_adults[dw_adults[col]]
    aud_total = sub.groupby('b_date')['weight'].sum().reindex(all_days, fill_value=0.0).mean()
    grp_daily = sub.groupby(['b_date', 'demo_group'])['weight'].sum().unstack(fill_value=0.0).reindex(all_days, fill_value=0.0)
    grp_means = grp_daily.mean()
    shares_dict[aud_name] = {g: grp_means.get(g, 0.0) / aud_total for g in group_order}

shares_df = pd.DataFrame(shares_dict).T[group_order]
index_df = shares_df.div(shares_df.loc['Adults 18+'], axis=1) * 100.0

profile_rows = []
for aud_name in audiences_map.keys():
    row_share = {'Audience': aud_name, 'Metric': 'Share'}
    row_idx = {'Audience': aud_name, 'Metric': 'Index'}
    for g in group_order:
        row_share[g] = f"{shares_df.loc[aud_name, g]:.3f}"
        row_idx[g] = f"{index_df.loc[aud_name, g]:.1f}"
    profile_rows.append(row_share)
    profile_rows.append(row_idx)

profile_table = pd.DataFrame(profile_rows)
display(profile_table)

,Audience,Metric,Men 18-34,Women 18-34,Men 35-54,Women 35-54,Men 55+,Women 55+
0,Adults 18+,Share,0.144,0.144,0.160,0.160,0.185,0.208
1,Adults 18+,Index,100.0,100.0,100.0,100.0,100.0,100.0
2,Shoe buyers,Share,0.140,0.140,0.158,0.178,0.186,0.197
3,Shoe buyers,Index,97.4,97.6,99.2,111.4,100.8,94.6
4,Heavy buyers (3+),Share,0.111,0.139,0.138,0.226,0.189,0.196
5,Heavy buyers (3+),Index,77.1,97.0,86.8,141.9,102.2,94.0
6,Kestrel buyers,Share,0.118,0.181,0.189,0.265,0.126,0.121
7,Kestrel buyers,Index,82.1,126.0,118.6,165.8,68.2,58.0
8,Adults 25-54,Share,0.174,0.174,0.326,0.326,0.000,0.000
9,Adults 25-54,Index,121.2,121.2,204.1,204.1,0.0,0.0


**Answer:** I would hand the planner heavy athletic-shoe buyers (3+ pairs) (49.6M adults, 1,661 panelists). It directly matches Kestrel's brief and targets frequent category spenders (including switchers from competing brands) with a large enough panel sample to rate programs reliably; by contrast, Kestrel buyers (10.0M) has only 354 panelists (making program-level ratings far too noisy and limiting the campaign to existing customers), shoe buyers (148.1M) is too broad, and Adults 25–54 (123.7M) misses the 38.5% of heavy buyers aged 55+ while sweeping in many non-buyers. "Athletic-shoe buyers" looks almost identical to all Adults 18+ (indexes between 94.6 and 111.4 across all six groups) because nearly 60% of adults (148.1M of 252.6M) bought at least one pair in the last 12 months—when a target covers most of the population, its demographic mix mathematically converges to the general adult average. Finally, because purchase behavior was fused onto the panel solely through demographics (age, sex, and household income), a fused attribute cannot reveal any true non-demographic affinity between a program's viewers and shoe buying (for example, whether sports or fitness-show viewers buy more running shoes than other people of the exact same age, sex, and income).

## Task 2.1 Programs by audience

In [18]:
# Fast exact average-minute rating calculator (timestamps strictly in int64 nanoseconds!)
person_viewings = viewings[viewings['resp_type'].isin(['R', 'L'])].copy()
v_start_ns_by_net = {}
v_end_ns_by_net = {}
v_resp_by_net = {}
v_weight_by_net = {}

for net, grp in person_viewings.groupby('network_code'):
    v_start_ns_by_net[net] = grp['start_ts'].astype('datetime64[ns]').astype('int64').to_numpy()
    v_end_ns_by_net[net] = grp['end_ts'].astype('datetime64[ns]').astype('int64').to_numpy()
    v_resp_by_net[net] = grp['respondent_id'].to_numpy()
    v_weight_by_net[net] = grp['weight'].to_numpy(dtype=np.float64)

adult_ids_arr = np.array(sorted(aud_flags.loc[aud_flags['is_adult'], 'respondent_id']))
heavy_ids_arr = np.array(sorted(aud_flags.loc[aud_flags['is_heavy'], 'respondent_id']))
kestrel_ids_arr = np.array(sorted(aud_flags.loc[aud_flags['is_kestrel'], 'respondent_id']))

daily_univ_p2 = persons_dw.groupby('b_date')['weight'].sum().to_dict()
daily_univ_adult = daily_adults.to_dict()
daily_univ_heavy = daily_heavy.to_dict()
daily_univ_kestrel = daily_kestrel.to_dict()

def rate_telecasts(telecasts_df, audience_specs):
    t_starts = telecasts_df['start_ts'].astype('datetime64[ns]').astype('int64').to_numpy()
    t_ends = telecasts_df['end_ts'].astype('datetime64[ns]').astype('int64').to_numpy()
    b_dates = pd.to_datetime(telecasts_df['broadcast_date']).dt.date.to_numpy()
    nets = telecasts_df['network_code'].to_numpy()
    t_ids = telecasts_df['telecast_id'].to_numpy()
    p_names = telecasts_df['program_name'].to_numpy()
    s_ts = telecasts_df['start_ts'].to_numpy()

    results = []
    for i in range(len(telecasts_df)):
        net = nets[i]
        t_start = t_starts[i]
        t_end = t_ends[i]
        t_min = (t_end - t_start) / 60e9
        b_date = b_dates[i]

        rec = {
            'telecast_id': t_ids[i],
            'network_code': net,
            'program_name': p_names[i],
            'broadcast_date': b_date,
            'start_ts': s_ts[i],
            'telecast_min': t_min
        }

        vs = v_start_ns_by_net.get(net)
        if vs is None or len(vs) == 0:
            for name in audience_specs:
                rec[f'imp_{name}'] = 0.0
                rec[f'rating_{name}'] = 0.0
                rec[f'n_pan_{name}'] = 0
            results.append(rec)
            continue

        ve = v_end_ns_by_net[net]
        mask = (vs < t_end) & (ve > t_start)
        if not np.any(mask):
            for name in audience_specs:
                rec[f'imp_{name}'] = 0.0
                rec[f'rating_{name}'] = 0.0
                rec[f'n_pan_{name}'] = 0
            results.append(rec)
            continue

        ov_start = np.maximum(vs[mask], t_start)
        ov_end = np.minimum(ve[mask], t_end)
        ov_min = (ov_end - ov_start) / 60e9
        resps = v_resp_by_net[net][mask]
        weights = v_weight_by_net[net][mask]
        w_min = weights * ov_min

        for name, (id_arr, univ_dict) in audience_specs.items():
            if id_arr is None:
                m_aud = np.ones(len(resps), dtype=bool)
            else:
                m_aud = np.isin(resps, id_arr)
            if np.any(m_aud):
                imp = w_min[m_aud].sum() / t_min
                rtg = (imp / univ_dict[b_date]) * 100.0
                n_pan = len(np.unique(resps[m_aud]))
            else:
                imp, rtg, n_pan = 0.0, 0.0, 0
            rec[f'imp_{name}'] = imp
            rec[f'rating_{name}'] = rtg
            rec[f'n_pan_{name}'] = n_pan
        results.append(rec)
    return pd.DataFrame(results)

# Price telecasts (:30) from rate_card (Lab 2 rules: program rows win over time rows)
is_prog_rc = rate_card['program_name'].notna() & (rate_card['program_name'].astype(str).str.strip() != '')
prog_rc = rate_card[is_prog_rc][['network_code', 'program_name', 'cost_30s_usd']].drop_duplicates(['network_code', 'program_name']).copy()
time_rc = rate_card[~is_prog_rc].copy()

def parse_bcast_min(t_str):
    h, m = map(int, str(t_str).strip().split(':'))
    return h * 60 + m - 360

time_rc['start_min'] = time_rc['start_time'].apply(parse_bcast_min)
time_rc['end_min'] = time_rc['end_time'].apply(parse_bcast_min)
time_rc_exp = time_rc.explode('weekdays').copy()
time_rc_exp['b_weekday'] = time_rc_exp['weekdays'].astype(int)

def price_telecasts_30s(telecasts_df):
    df = telecasts_df[['telecast_id', 'network_code', 'program_name', 'start_ts']].copy()
    df = df.merge(prog_rc.rename(columns={'cost_30s_usd': 'prog_cost'}), on=['network_code', 'program_name'], how='left')
    unmatched = df[df['prog_cost'].isna()].copy()
    if len(unmatched) > 0:
        ts_shifted = unmatched['start_ts'] - pd.Timedelta(hours=6)
        unmatched['b_weekday'] = ts_shifted.dt.dayofweek
        unmatched['t_min'] = ts_shifted.dt.hour * 60 + ts_shifted.dt.minute + ts_shifted.dt.second / 60.0
        mt = unmatched.merge(
            time_rc_exp[['network_code', 'b_weekday', 'start_min', 'end_min', 'cost_30s_usd']],
            on=['network_code', 'b_weekday'],
            how='left'
        )
        mt = mt[(mt['t_min'] >= mt['start_min']) & (mt['t_min'] < mt['end_min'])].drop_duplicates('telecast_id')
        df = df.merge(mt[['telecast_id', 'cost_30s_usd']].rename(columns={'cost_30s_usd': 'time_cost'}), on='telecast_id', how='left')
    else:
        df['time_cost'] = np.nan
    df['cost_30s_usd'] = df['prog_cost'].fillna(df['time_cost'])
    return df[['telecast_id', 'cost_30s_usd']]

# Select the 8 Prime-time programs
prime_progs = programs[programs['daypart'] == 'Prime'].copy()
task2_defs = [
    ('Sunday Night Football (NBC)', 'NBC', 'NFL Football'),
    ('Monday Night Football (ABC)', 'ABC', 'NFL Football'),
    ('Dancing with the Stars (ABC)', 'ABC', 'Dancing with the Stars'),
    ('NCIS (CBS)', 'CBS', 'NCIS'),
    ('Hannity (FNC)', 'FNC', 'Hannity'),
    ('Unsellable Houses (HGTV)', 'HGTV', 'Unsellable Houses'),
    ('Halloween Wars (FOOD)', 'FOOD', 'Halloween Wars'),
    ('Grotesquerie (FX)', 'FX', 'Grotesquerie'),
]

selected_list = []
for label, net, pname in task2_defs:
    sub = prime_progs[(prime_progs['network_code'] == net) & (prime_progs['program_name'] == pname)].copy()
    sub['display_name'] = label
    selected_list.append(sub)
task2_telecasts = pd.concat(selected_list, ignore_index=True)

aud_specs_t2 = {
    'adult': (adult_ids_arr, daily_univ_adult),
    'heavy': (heavy_ids_arr, daily_univ_heavy),
    'kestrel': (kestrel_ids_arr, daily_univ_kestrel),
}
t2_rated = rate_telecasts(task2_telecasts, aud_specs_t2)
t2_priced = price_telecasts_30s(task2_telecasts)
t2_full = t2_rated.merge(task2_telecasts[['telecast_id', 'display_name']], on='telecast_id').merge(t2_priced, on='telecast_id')

t2_agg = (
    t2_full.groupby('display_name', sort=False)
    .agg(
        telecasts=('telecast_id', 'count'),
        adults_rating=('rating_adult', 'mean'),
        target_rating=('rating_heavy', 'mean'),
        kestrel_rating=('rating_kestrel', 'mean'),
        adults_viewers=('imp_adult', 'mean'),
        target_viewers=('imp_heavy', 'mean'),
        cost_30s=('cost_30s_usd', 'mean'),
        mean_heavy_pan=('n_pan_heavy', 'mean'),
        mean_kestrel_pan=('n_pan_kestrel', 'mean'),
        min_kestrel_pan=('n_pan_kestrel', 'min'),
    )
    .reset_index()
)

t2_agg['target_viewers_k'] = t2_agg['target_viewers'] / 1000.0
t2_agg['composition'] = t2_agg['target_viewers'] / t2_agg['adults_viewers']
t2_agg['index'] = (t2_agg['target_rating'] / t2_agg['adults_rating']) * 100.0
t2_agg['kestrel_index'] = (t2_agg['kestrel_rating'] / t2_agg['adults_rating']) * 100.0
t2_agg['adults_cpm'] = (t2_agg['cost_30s'] / t2_agg['adults_viewers']) * 1000.0
t2_agg['target_cpm'] = (t2_agg['cost_30s'] / t2_agg['target_viewers']) * 1000.0

t2_table = pd.DataFrame({
    'Program': t2_agg['display_name'],
    'Adults rating': t2_agg['adults_rating'].round(2),
    'Target rating': t2_agg['target_rating'].round(2),
    'Target viewers (K)': t2_agg['target_viewers_k'].round(1),
    'Composition': t2_agg['composition'].round(3),
    'Index': t2_agg['index'].round(1),
    'Cost per :30 ($)': t2_agg['cost_30s'].round(0).astype(int),
    'Adults CPM ($)': t2_agg['adults_cpm'].round(2),
    'Target CPM ($)': t2_agg['target_cpm'].round(2),
})

display(t2_table)
print("Target universe (M):", f"{size_heavy / 1e6:.1f}")

,Program,Adults rating,Target rating,Target viewers (K),Composition,Index,Cost per :30 ($),Adults CPM ($),Target CPM ($)
0,Sunday Night Football (NBC),5.82,5.79,2865.0,0.195,99.3,487200,33.12,170.05
1,Monday Night Football (ABC),8.93,8.36,4149.1,0.184,93.6,723600,32.09,174.40
2,Dancing with the Stars (ABC),3.25,3.14,1564.2,0.191,96.8,158600,19.33,101.39
3,NCIS (CBS),2.21,2.43,1206.3,0.216,110.1,178800,32.09,148.22
4,Hannity (FNC),1.09,0.84,416.9,0.152,77.1,39900,14.50,95.71
5,Unsellable Houses (HGTV),0.50,0.64,317.2,0.249,126.8,12900,10.12,40.66
6,Halloween Wars (FOOD),0.28,0.33,161.9,0.228,116.2,9700,13.67,59.90
7,Grotesquerie (FX),0.26,0.25,124.2,0.188,95.9,7592,11.50,61.11


Target universe (M): 49.6


## Task 2.2 Index, volume or price?

In [19]:
highest_idx_prog = t2_agg.loc[t2_agg['index'].idxmax(), 'display_name']
most_viewers_prog = t2_agg.loc[t2_agg['target_viewers'].idxmax(), 'display_name']
lowest_cpm_prog = t2_agg.loc[t2_agg['target_cpm'].idxmin(), 'display_name']

print("Highest index (program):", highest_idx_prog)
print("Most target viewers (program):", most_viewers_prog)
print("Lowest target CPM (program):", lowest_cpm_prog)

Highest index (program): Unsellable Houses (HGTV)
Most target viewers (program): Monday Night Football (ABC)
Lowest target CPM (program): Unsellable Houses (HGTV)


**Answer:** They are not the same program: Unsellable Houses (HGTV) has both the highest index (126.8) and the lowest target CPM (\$40.66), whereas Monday Night Football (ABC) delivers the most target viewers per spot (4,149.1K—over 13× the 317.2K of Unsellable Houses). I would tell a client who insists on buying "only the highest index" that index alone ignores both scale and price (just like Lecture 5's shampoo example): a high-index cable show reaches only a small slice of the target (0.64 rating), so restricting a buy to top-indexing programs caps reach and piles up frequency on the same small audience while passing over efficient or high-reach alternatives. For example, Grotesquerie (FX) indexes below 100 (95.9) yet costs just \$61.11 per thousand heavy buyers—less than half the target CPM of higher-indexing NCIS (CBS) (index 110.1, target CPM \$148.22)—and broad broadcast shows are still needed when a campaign requires rapid volume and reach; the planner should optimize on target CPM (eCPM) and reach together, not index alone.

## Task 2.3 Kestrel buyers

In [20]:
t23_table = pd.DataFrame({
    'Program': t2_agg['display_name'],
    'Heavy-buyer index': t2_agg['index'].round(1),
    'Kestrel index': t2_agg['kestrel_index'].round(1),
    'Avg heavy panelists': t2_agg['mean_heavy_pan'].round(1),
    'Avg Kestrel panelists': t2_agg['mean_kestrel_pan'].round(1),
})
display(t23_table)

,Program,Heavy-buyer index,Kestrel index,Avg heavy panelists,Avg Kestrel panelists
0,Sunday Night Football (NBC),99.3,98.5,329.0,75.2
1,Monday Night Football (ABC),93.6,85.9,292.7,69.3
2,Dancing with the Stars (ABC),96.8,82.7,122.6,23.4
3,NCIS (CBS),110.1,56.5,86.3,11.0
4,Hannity (FNC),77.1,42.6,15.9,2.1
5,Unsellable Houses (HGTV),126.8,76.9,25.4,4.0
6,Halloween Wars (FOOD),116.2,95.2,13.7,2.8
7,Grotesquerie (FX),95.9,129.6,11.4,3.1


**Answer:** The Kestrel index jumps dramatically—from 42.6 on Hannity and 56.5 on NCIS to 129.6 on Grotesquerie, and even dropping to 76.9 on Unsellable Houses where heavy buyers index 126.8—because of extreme sampling noise from a tiny in-target panel base. There are only 354 Kestrel buyers in the entire panel (versus 1,661 heavy buyers), leaving an average of just 2.1 to 4.0 Kestrel panelists watching per telecast on the four cable shows and 11.0 on NCIS; when a rating rests on 2–4 people, a single panelist tuning in or out shifts the index by dozens of points. I would trust the heavy-buyer index (especially on broadcast shows with 86–329 heavy panelists, and directionally on cable with 11–25), and distrust the Kestrel index everywhere except possibly the two NFL telecasts (~69–75 Kestrel panelists). To verify stability, I would compute the relative standard error / confidence interval from unweighted panelist counts, check split-half (first half vs. second half of October) or episode-to-episode consistency, and compare the observed index against the demographic-expected index implied by the fusion.

## Task 3.1 Weights

In [21]:
# 1. Convert ACR UTC timestamps to US Eastern (naive datetime64[ns], per Lab 3 rules)
acr = acr_sessions.copy()
acr['start_et'] = pd.to_datetime(acr['start_utc'], utc=True).dt.tz_convert('America/New_York').dt.tz_localize(None).astype('datetime64[ns]')
acr['end_et'] = pd.to_datetime(acr['end_utc'], utc=True).dt.tz_convert('America/New_York').dt.tz_localize(None).astype('datetime64[ns]')

# 2. 1-minute edit rule (>= 60 seconds) and broadcast day (06:00-05:59 ET)
acr['duration_sec'] = (acr['end_et'] - acr['start_et']).dt.total_seconds()
acr_clean = acr[acr['duration_sec'] >= 60.0].copy()
acr_clean['b_date'] = (acr_clean['start_et'] - pd.Timedelta(hours=6)).dt.date

# 3. Match TVs to exactly one household via hashed_ip (Lab 3 rule)
tv_ips = acr[['tv_id', 'hashed_ip']].dropna().drop_duplicates()
tv_hh_links = tv_ips.merge(
    hh_attributes[['household_id', 'hashed_ip', 'head_age_band', 'income_band']],
    on='hashed_ip',
    how='inner'
)
tv_hh_counts = tv_hh_links.groupby('tv_id')['household_id'].nunique()
matched_1hh_tvs = set(tv_hh_counts[tv_hh_counts == 1].index)

tv_meta = (
    tv_hh_links[tv_hh_links['tv_id'].isin(matched_1hh_tvs)]
    .drop_duplicates('tv_id')[['tv_id', 'household_id', 'head_age_band', 'income_band']]
    .copy()
)

# Map head_age_band to universe head_age ('65+' vs 'Under 65')
tv_meta['head_age'] = np.where(tv_meta['head_age_band'].astype(str).str.contains('65'), '65+', 'Under 65')
tv_meta = tv_meta.merge(universe[['head_age', 'income_band', 'tv_households']], on=['head_age', 'income_band'], how='left')

# 4. Daily in-tab matched TVs & cell weights
acr_matched = acr_clean[acr_clean['tv_id'].isin(matched_1hh_tvs)].copy()
intab_tv_days = (
    acr_matched[['tv_id', 'b_date']].drop_duplicates()
    .merge(tv_meta[['tv_id', 'head_age', 'income_band', 'tv_households']], on='tv_id', how='left')
)

cell_day_counts = (
    intab_tv_days.groupby(['b_date', 'head_age', 'income_band'])['tv_id']
    .nunique()
    .reset_index(name='cell_tvs')
)
intab_tv_days = intab_tv_days.merge(cell_day_counts, on=['b_date', 'head_age', 'income_band'], how='left')
intab_tv_days['weight'] = intab_tv_days['tv_households'] / intab_tv_days['cell_tvs']

daily_weight_sums = intab_tv_days.groupby('b_date')['weight'].sum()
weights_sum_per_day_m = daily_weight_sums.mean() / 1e6

# 5. Compute unweighted & weighted ratings for the 8 reference_ratings telecasts (Lab 3 rules: live, overlap >= 60s)
ref_merged = reference_ratings.merge(
    programs[['telecast_id', 'network_code', 'program_name', 'broadcast_date', 'start_ts', 'end_ts']],
    on='telecast_id',
    how='left',
    suffixes=('', '_prog')
)
ref_val_col = 'hh_rating' if 'hh_rating' in ref_merged.columns else [c for c in reference_ratings.columns if 'rating' in c.lower()][0]

def compute_ref_ratings(acr_df, session_types=('live',)):
    acr_v = acr_df[acr_df['session_type'].isin(session_types)].copy()
    av_s = acr_v['start_et'].astype('datetime64[ns]').astype('int64').to_numpy()
    av_e = acr_v['end_et'].astype('datetime64[ns]').astype('int64').to_numpy()
    av_net = acr_v['network_code'].to_numpy()
    av_tv = acr_v['tv_id'].to_numpy()

    rows = []
    for r in ref_merged.itertuples(index=False):
        b_date = pd.to_datetime(r.broadcast_date).date()
        net = r.network_code
        t_start_ns = pd.Timestamp(r.start_ts).as_unit('ns').value
        t_end_ns = pd.Timestamp(r.end_ts).as_unit('ns').value

        day_intab = intab_tv_days[intab_tv_days['b_date'] == b_date]
        total_intab_tvs = len(day_intab)
        total_intab_w = day_intab['weight'].sum()

        m = (av_net == net) & (av_s < t_end_ns) & (av_e > t_start_ns)
        if np.any(m):
            ov_sec = (np.minimum(av_e[m], t_end_ns) - np.maximum(av_s[m], t_start_ns)) / 1e9
            watched_tvs = set(av_tv[m][ov_sec >= 60.0])
        else:
            watched_tvs = set()

        watched_intab = day_intab[day_intab['tv_id'].isin(watched_tvs)]
        unw_rtg = (len(watched_intab) / total_intab_tvs) * 100.0
        w_rtg = (watched_intab['weight'].sum() / total_intab_w) * 100.0
        ref_rtg = getattr(r, ref_val_col)

        rows.append({
            'Telecast': f"{r.program_name} ({net}) {b_date}",
            'Unweighted': round(unw_rtg, 2),
            'Weighted': round(w_rtg, 2),
            'Reference': round(ref_rtg, 2),
            'Abs_gap_unw': abs(unw_rtg - ref_rtg),
            'Abs_gap_w': abs(w_rtg - ref_rtg),
        })
    return pd.DataFrame(rows)

t3_res = compute_ref_ratings(acr_matched)
mean_gap_unw = t3_res['Abs_gap_unw'].mean()
mean_gap_w = t3_res['Abs_gap_w'].mean()
share_gap_closed = (mean_gap_unw - mean_gap_w) / mean_gap_unw

print("TVs matched to one household:", len(matched_1hh_tvs))
print("Weights sum per day (M):", f"{weights_sum_per_day_m:.1f}")
display(t3_res[['Telecast', 'Unweighted', 'Weighted', 'Reference']])
print("Mean gap unweighted (points):", f"{mean_gap_unw:.2f}")
print("Mean gap weighted (points):", f"{mean_gap_w:.2f}")
print("Share of gap closed:", f"{share_gap_closed:.3f}")

TVs matched to one household: 19607
Weights sum per day (M): 125.8


,Telecast,Unweighted,Weighted,Reference
0,MLB Postseason (FOX) 2024-10-25,17.03,17.76,25.86
1,NFL Football (NBC) 2024-10-27,22.62,23.75,33.14
2,NCIS (CBS) 2024-10-14,5.50,5.80,10.56
3,Hannity (FNC) 2024-10-16,1.04,1.35,2.30
4,Unsellable Houses (HGTV) 2024-10-16,2.12,2.08,3.69
5,Halloween Baking Championship (FOOD) 2024-10-14,0.95,0.90,1.63
6,Grotesquerie (FX) 2024-10-16,2.17,2.11,3.53
7,The Real Housewives of New York City (BRAVO) 2024-10-15,0.71,0.64,1.23


Mean gap unweighted (points): 3.73
Mean gap weighted (points): 3.44
Share of gap closed: 0.075


## Task 3.2 What weighting cannot fix

**Answer:** Weighting moved the four older-skewing broadcast and cable-news telecasts up toward the reference—NFL Football (NBC) (+1.13 pts), MLB Postseason (FOX) (+0.72 pts), Hannity (FNC) (+0.31 pts, a ~30% relative lift), and NCIS (CBS) (+0.29 pts)—while nudging the four younger-skewing cable entertainment shows (RHONY, Grotesquerie, Halloween Baking Championship, and Unsellable Houses) slightly down (-0.04 to -0.07 pts). This happens because the matched smart-TV footprint severely under-represents 65+ households (19.4% of matched TVs vs. 28.3% of the US TV universe), so up-weighting 65+ homes lifts programs popular with older viewers and trims younger-skewing cable shows. Yet weighting closed only 7.5% (0.075) of the mean gap (3.73 to 3.44 points) because weighting on age and income cannot fix structural measurement and unmeasured selection biases: an ACR feed observes a single opted-in smart TV rather than all TVs in the household, misses unrecognized (`unknown`) or unmonitored linear tuning, excludes non-smart-TV and opt-out homes (plus households dropped in IP matching), and over-samples heavier streaming-app users who watch less linear TV even within the same age-income cell. Weighting by income barely moves these ratings for two reasons: the footprint's income skew is modest (within ~3–4 percentage points per band vs. an ~9-point gap in 65+), and linear tuning for these programs correlates much more strongly with age than with household income.

## Task 4.1 October predicts October

In [22]:
big5_nets = ['CBS', 'NBC', 'ABC', 'FOX', 'FNC']
p4 = programs[(programs['daypart'] == 'Prime') & (programs['network_code'].isin(big5_nets))].copy()
p4['b_date'] = pd.to_datetime(p4['broadcast_date']).dt.date

hist_start, hist_end = pd.to_datetime('2024-10-01').date(), pd.to_datetime('2024-10-21').date()
fcst_start, fcst_end = pd.to_datetime('2024-10-22').date(), pd.to_datetime('2024-10-28').date()

p4_period = p4[(p4['b_date'] >= hist_start) & (p4['b_date'] <= fcst_end)].copy()

aud_specs_t4 = {
    'p2': (None, daily_univ_p2),
    'heavy': (heavy_ids_arr, daily_univ_heavy),
}
p4_rated = rate_telecasts(p4_period, aud_specs_t4)

hist_rated = p4_rated[(p4_rated['broadcast_date'] >= hist_start) & (p4_rated['broadcast_date'] <= hist_end)].copy()
fcst_rated = p4_rated[(p4_rated['broadcast_date'] >= fcst_start) & (p4_rated['broadcast_date'] <= fcst_end)].copy()

hist_means = (
    hist_rated.groupby(['network_code', 'program_name'])
    .agg(
        fcst_p2=('rating_p2', 'mean'),
        fcst_heavy=('rating_heavy', 'mean'),
        hist_n=('telecast_id', 'count')
    )
    .reset_index()
)

fcst_eval = fcst_rated.merge(hist_means, on=['network_code', 'program_name'], how='inner')

eval_p2 = fcst_eval[fcst_eval['rating_p2'] > 0].copy()
eval_p2['ape_p2'] = (eval_p2['rating_p2'] - eval_p2['fcst_p2']).abs() / eval_p2['rating_p2'] * 100.0

eval_heavy = fcst_eval[fcst_eval['rating_heavy'] > 0].copy()
eval_heavy['ape_heavy'] = (eval_heavy['rating_heavy'] - eval_heavy['fcst_heavy']).abs() / eval_heavy['rating_heavy'] * 100.0

mape_p2 = eval_p2['ape_p2'].mean()
mape_heavy = eval_heavy['ape_heavy'].mean()

biggest_row = eval_p2.sort_values('ape_p2', ascending=False).iloc[0]
biggest_miss_str = f"{biggest_row['program_name']} ({biggest_row['network_code']}), {biggest_row['broadcast_date'].strftime('%Y-%m-%d')}"

print("Telecasts forecast (n):", len(eval_p2))
print("MAPE, P2+ (%):", f"{mape_p2:.1f}")
print("MAPE, heavy buyers (%):", f"{mape_heavy:.1f}")
print("Biggest miss, P2+ (program, date):", biggest_miss_str)

Telecasts forecast (n): 101
MAPE, P2+ (%): 35.8
MAPE, heavy buyers (%): 38.6
Biggest miss, P2+ (program, date): Krapopolis (FOX), 2024-10-27


## Task 4.2 Why it missed

In [23]:
top5_misses = eval_p2.sort_values('ape_p2', ascending=False).head(5).copy()
top5_table = pd.DataFrame({
    'Program': top5_misses['program_name'] + ' (' + top5_misses['network_code'] + ')',
    'Date': top5_misses['broadcast_date'].astype(str),
    'Forecast': top5_misses['fcst_p2'].round(2),
    'Actual': top5_misses['rating_p2'].round(2),
    'APE (%)': top5_misses['ape_p2'].round(1)
}).reset_index(drop=True)
display(top5_table)

,Program,Date,Forecast,Actual,APE (%)
0,Krapopolis (FOX),2024-10-27,1.06,0.22,387.8
1,Universal Basic Guys (FOX),2024-10-27,0.68,0.26,162.2
2,Fire Country (CBS),2024-10-27,1.70,0.67,154.8
3,Accused (FOX),2024-10-22,0.95,0.37,154.6
4,What Would You Do? (ABC),2024-10-23,0.44,0.18,138.1


**Answer:** The three biggest misses all occurred on Sunday, October 27, 2024: Fire Country (CBS) (forecast 1.70 vs. actual 0.67) was pushed to a late 22:44 start after an NFL overrun shifted CBS's entire prime-time lineup (60 Minutes started at 19:35, Tracker at 20:44), while FOX's Krapopolis (1.06 vs. 0.22) and Universal Basic Guys (0.68 vs. 0.26) also followed an off-clock 19:15 local block and faced heavy Sunday competition from NBC's NFL Football (5.36) and ABC's Dancing with the Stars (4.53). The other two misses reflect time-slot/lead-in differences amplified by small denominators in MAPE: Accused (FOX) rated just 0.37 on Tuesday 2024-10-22 at 21:00 opposite FBI: International, The Voice, and DWTS (compared to 2.14 when it aired Wednesday at 22:00), and What Would You Do? (ABC) fell to 0.18 on 2024-10-23 behind weak ABC lead-ins against Accused and Chicago P.D.. The heavy-buyer forecast is less accurate than P2+ (38.6% vs. 35.8% MAPE) because fewer in-target panelists watch each telecast (an average of 62.7 heavy buyers vs. 348.1 P2+ panelists), increasing sampling variance in both the historical mean and the actual weekly rating (Lecture 6). A better forecast would incorporate (1) schedule and competitive context (day of week, exact time slot, lead-in program rating, and major competing live sports or specials) and (2) airing attributes and trend (first-run vs. repeat flags, special-event indicators, and recency-weighted or shrinkage-adjusted estimates).

## Task 5.1 The plan

In [25]:
spot_allocation = {
    'Dancing with the Stars (ABC)': 2,
    'NCIS (CBS)': 1,
    'Unsellable Houses (HGTV)': 20,
    'Halloween Wars (FOOD)': 14,
    'Grotesquerie (FX)': 14,
}

plan_rows = []
for prog, spots in spot_allocation.items():
    r = t2_agg.loc[t2_agg['display_name'] == prog].iloc[0]
    spot_cost = int(round(r['cost_30s']))
    cost = spots * spot_cost
    imp = spots * r['target_viewers']
    adult_imp = spots * r['adults_viewers']
    plan_rows.append({
        'Program': prog,
        'Spots': spots,
        'Cost ($)': cost,
        'Target impressions (M)': round(imp / 1e6, 2),
        '_imp': imp,
        '_adult_imp': adult_imp,
    })

plan_df = pd.DataFrame(plan_rows)
display(plan_df[['Program', 'Spots', 'Cost ($)', 'Target impressions (M)']])

plan_cost = int(plan_df['Cost ($)'].sum())
plan_target_imp_m = plan_df['_imp'].sum() / 1e6
blended_comp = plan_df['_imp'].sum() / plan_df['_adult_imp'].sum()

print("Plan cost ($):", plan_cost)
print("Plan target impressions (M):", f"{plan_target_imp_m:.1f}")
print("Blended competition:", f"{blended_comp:.2f}")

,Program,Spots,Cost ($),Target impressions (M)
0,Dancing with the Stars (ABC),2,317200,3.13
1,NCIS (CBS),1,178800,1.21
2,Unsellable Houses (HGTV),20,258000,6.34
3,Halloween Wars (FOOD),14,135800,2.27
4,Grotesquerie (FX),14,106288,1.74


Plan cost ($): 996088
Plan target impressions (M): 14.7
Blended competition: 0.22


## Task 5.2 Recommendation

**Answer:** For Kestrel's \$1,000,000 one-week November flight, I recommend spending \$996,088 across five programs—2 spots in Dancing with the Stars (ABC) (\$317,200), 1 spot in NCIS (CBS) (\$178,800), 20 spots in Unsellable Houses (HGTV) (\$258,000), 14 spots in Halloween Wars (FOOD) (\$135,800), and 14 spots in Grotesquerie (FX) (\$106,288)—to deliver an estimated 14.7 million heavy-buyer impressions at a blended target CPM of \$67.83. This mix pairs broad prime-time broadcast reach from Dancing with the Stars (\$101.39 target CPM) and high-indexing NCIS (110.1 index, \$148.22 target CPM) with high-efficiency cable inventory costing \$40.66–\$61.11 per thousand heavy buyers and indexing up to 126.8. We pass on Sunday Night Football (\$170.05 target CPM) and Monday Night Football (\$174.40) because one spot would absorb half to three-quarters of the budget at over 2.5× the blended CPM, and we drop Hannity due to its weak 77.1 index. Delivery carries substantial uncertainty—our October backtest showed a 38.6% per-telecast MAPE for heavy buyers due to schedule shifts, small in-target panel samples, and November's Election Week volatility—though spreading 51 spots across five networks diversifies part of that variance. Because under-delivery risk is high, we should negotiate guaranteed audience delivery on the broadcast commitments so the networks owe make-goods if ratings fall short. Even with this targeted mix, the plan's blended composition is 0.220, meaning 78.0% of impressions (roughly \$777,000 of spend) will reach adults who are not heavy athletic-shoe buyers—an unavoidable linear-TV trade-off that still builds broader brand awareness. After the week airs, we will run a post-buy report on the November panel to verify actual heavy-buyer impressions, reach and frequency, and realized target CPM against guarantees (claiming make-goods where owed), alongside tracking Kestrel web traffic and retail sales lift.